# RL Casino — Sparse BSR smoke test (Colab)

This notebook is meant to be **copy/pasted into Colab** to quickly validate:
- Triton compilation works on the GPU runtime
- `SparseMaskManager` + `SparseAdamW` can run a small update without crashing
- You can toggle the optimizer kernel mode (`block_1d` vs `block_2d`) and compare timing

It does **not** run full DPO training. It’s just a fast “will this compile + step” gate.


In [ ]:
# If you're running in a fresh Colab runtime
!nvidia-smi

# Install Triton and a CUDA-enabled PyTorch.
# Colab usually already has torch+cuda; keep this minimal unless you hit import errors.
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U"] + pkgs)

pip_install(["triton"])

In [ ]:
# Option A: clone your repo (recommended).
# If your repo is private, use a Colab secret or manual auth.
# REPO_URL = "https://github.com/<you>/<repo>.git"
# !git clone "$REPO_URL" rl_casino

# Option B: if you already uploaded the repo as a zip / mounted Drive, set REPO_ROOT accordingly.
import os
REPO_ROOT = os.path.abspath(".")  # change to /content/rl_casino if you cloned it
print("REPO_ROOT=", REPO_ROOT)

# Ensure `src` imports work
import sys
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import torch
print("torch", torch.__version__)
print("cuda?", torch.cuda.is_available())
assert torch.cuda.is_available(), "Please switch Colab runtime to GPU: Runtime → Change runtime type → GPU"

In [ ]:
import time
import tempfile
from pathlib import Path

from src.utils.mask_manager import SparseMaskManager
from src.optimizers.sparse_adamw import SparseAdamW

device = torch.device("cuda")

# Tuning knobs
DTYPE = torch.bfloat16
M, N = 2048, 2048  # keep modest for Colab (increase if you want)
SPARSITY = 0.9975  # fraction of zeros (True means active weight)
BLOCK_SIZE_BSR = 16

def make_mask_bool(m, n, sparsity, *, seed=0):
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))
    # mask True = keep
    keep_prob = 1.0 - float(sparsity)
    mask = (torch.rand((m, n), generator=g) < keep_prob)
    return mask.to(torch.bool)

mask = make_mask_bool(M, N, SPARSITY, seed=123)
print("mask keep%", 100.0 * mask.float().mean().item())

# Save a minimal .pt mask file in the format SparseMaskManager expects.
param_name = "toy.weight"
mask_payload = {"masks": {param_name: mask}}

tmp = tempfile.NamedTemporaryFile(suffix=".pt", delete=False)
tmp_path = Path(tmp.name)
tmp.close()
torch.save(mask_payload, str(tmp_path))
print("mask_path=", tmp_path)

# Build toy parameter + gradient
p = torch.nn.Parameter(torch.randn((M, N), device=device, dtype=DTYPE) * 0.01)
p.grad = torch.randn_like(p) * 0.01

mm = SparseMaskManager(str(tmp_path), device=device)

# NOTE: SparseAdamW chooses block kernels when active blocks exist.
# We benchmark block_1d vs block_2d by toggling RL_CASINO_ADAM_KERNEL.

opt = SparseAdamW([(param_name, p)], mm, lr=1e-3, weight_decay=0.01, block_size=128, mlp_only=False, max_grad_norm=0.0)
torch.cuda.synchronize()

In [ ]:
def bench(kernel_mode: str, iters: int = 10, warmup: int = 3):
    import os
    os.environ["RL_CASINO_ADAM_KERNEL"] = kernel_mode
    # fresh grad each iter to avoid degenerate timings
    for _ in range(warmup):
        p.grad = torch.randn_like(p) * 0.01
        opt.step()
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        p.grad = torch.randn_like(p) * 0.01
        opt.step()
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    print(f"{kernel_mode}: {dt*1e3:.3f} ms/step")

bench("block_1d")
bench("block_2d")

## Notes / troubleshooting

- If Triton compilation fails, print the full traceback and capture the first error; it’s often a missing CUDA toolkit component in the runtime.
- If you see OOM, lower `(M, N)`.
- If timings are too noisy, increase `iters` to 50.
